In [2]:
import requests 
import pandas as pd
from bs4 import BeautifulSoup 

In [ ]:
def get_all(url):
    # Define the base URL used to construct complete GitHub URLs
    base_url = 'https://github.com'
    # Set the maximum wait time for each HTTP request to 15 seconds
    timeout = 15

    # Send a GET request to the GitHub Topics page
    r = requests.get(url,timeout=timeout)

    # Check whether the request was successful (HTTP status code 200)
    # If not, raise an exception and stop execution
    if r.status_code !=200:
        raise Exception(f"Faild to load page{url}")

    # Parse the HTML response into a BeautifulSoup object
    soup = BeautifulSoup(r.text,'html.parser')

    all_repo = soup.select_one('div.col-lg-9.position-relative')
    inner_repo = all_repo.select('div.tmp-py-4.border-bottom.d-flex.flex-justify-between')

    # Create an empty list to store the URLs of individual topic pages
    urls  = [] 
    # Iterate through each topic element found on the Topics page
    for each in inner_repo:
        urls.append(base_url+each.select_one('a.no-underline.flex-1.d-flex.flex-column')['href']) 

    author_name=[]
    repo_title=[]
    repo_url=[] 
    author_acc=[]
    star_count=[]

    # Visit each topic page URL collected above
    for u in urls:
        each_url = requests.get(u,timeout=timeout)
        if each_url.status_code !=200:
                raise Exception(f"Faild to load page{each_url}")
        topic_doc = BeautifulSoup(each_url.text,'html.parser')

        outer = topic_doc.select('div.d-flex.flex-justify-between.flex-items-start.flex-wrap.gap-2.tmp-my-3')  

        # Iterate through each repository card found on this topic page
        for repo_card in outer: 
           
           name=repo_card.find('a',class_='Link').text.strip()
           title=repo_card.find('a',class_='Link text-bold wb-break-word').text.strip()
           star=repo_card.find('span',class_='Counter js-social-count').text
           star=int(float(star.replace('k',''))*1000)
           repository=repo_card.find('a',class_='Link text-bold wb-break-word')['href']
           repository = base_url+repository
           author_account=repo_card.find('a',class_='Link text-bold wb-break-word')['href']
           author_account = base_url+author_account

           author_name.append(name)
           repo_title.append(title)
           star_count.append(star)
           repo_url.append(repository)
           author_acc.append(author_account) 

    # Combine all extracted lists into a dictionary,
    # where each key represents a DataFrame column
    dict = {
        'author_name':author_name,
        'repository':repo_title,
        'author_account':author_acc,
        'repository_link':repo_url,
        'star':star_count 
    } 

    # Convert the dictionary into a Pandas DataFrame
    # and return the final structured dataset
    return pd.DataFrame(dict) 

In [4]:
url='https://github.com/topics'

try:
    complete_df = get_all(url)
except Exception as e:
    print(e)  

In [6]:
complete_df.head(5)

,author_name,repository,author_account,repository_link,star
0,sindresorhus,awesome,https://github.com/sindresorhus/awesome,https://github.com/sindresorhus/awesome,514000
1,vinta,awesome-python,https://github.com/vinta/awesome-python,https://github.com/vinta/awesome-python,325000
2,awesome-selfhosted,awesome-selfhosted,https://github.com/awesome-selfhosted/awesome-...,https://github.com/awesome-selfhosted/awesome-...,324000
3,trimstray,the-book-of-secret-knowledge,https://github.com/trimstray/the-book-of-secre...,https://github.com/trimstray/the-book-of-secre...,248000
4,avelino,awesome-go,https://github.com/avelino/awesome-go,https://github.com/avelino/awesome-go,187000
